# Phase 3: Model Registry & Deployment

We take the trained model from GCS and:
1. Upload it to the Vertex AI Model Registry — a versioned catalog of models
2. Create an Endpoint — a live REST API for serving predictions
3. Deploy the model to the endpoint
4. Send test predictions to confirm it works

This is where the model becomes a live service rather than a file.

In [1]:
from google.cloud import aiplatform

PROJECT_ID = "churn-mlops-498321"
BUCKET_NAME = "churn-mlops-bucket-caston"
REGION = "us-central1"

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=f"gs://{BUCKET_NAME}"
)
print("Vertex AI initialized")

Vertex AI initialized


In [2]:
# Upload the model to the Vertex AI Model Registry
# artifact_uri points to the FOLDER containing model.pkl (not the file itself)
# The serving_container is a pre-built image that knows how to serve sklearn models
model = aiplatform.Model.upload(
    display_name="churn-rf-model",
    artifact_uri=f"gs://{BUCKET_NAME}/models",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest",
    description="Random Forest churn classifier, ROC-AUC 0.81"
)

print(f"Model registered: {model.resource_name}")
print(f"Model display name: {model.display_name}")

Creating Model
Create Model backing LRO: projects/434019816081/locations/us-central1/models/4404415432207892480/operations/8268938966909059072
Model created. Resource name: projects/434019816081/locations/us-central1/models/4404415432207892480@1
To use this Model in another session:
model = aiplatform.Model('projects/434019816081/locations/us-central1/models/4404415432207892480@1')
Model registered: projects/434019816081/locations/us-central1/models/4404415432207892480
Model display name: churn-rf-model


In [4]:
# The endpoint was already created, so reference it directly and deploy to it
endpoint = aiplatform.Endpoint('projects/434019816081/locations/us-central1/endpoints/5740286875583643648')

endpoint.deploy(
    model=model,
    deployed_model_display_name="churn-rf-deployed",
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=1,
    sync=True
)

print("Model deployed successfully")
print(f"Endpoint: {endpoint.resource_name}")

Deploying Model projects/434019816081/locations/us-central1/models/4404415432207892480 to Endpoint : projects/434019816081/locations/us-central1/endpoints/5740286875583643648
Deploy Endpoint model backing LRO: projects/434019816081/locations/us-central1/endpoints/5740286875583643648/operations/6938836661044248576
Endpoint model deployed. Resource name: projects/434019816081/locations/us-central1/endpoints/5740286875583643648
Model deployed successfully
Endpoint: projects/434019816081/locations/us-central1/endpoints/5740286875583643648


## Test the Live Endpoint
Now that the model is deployed, we send real customer records from our
test set to the endpoint and get live predictions back. This confirms
the endpoint is serving correctly.

The endpoint expects the same 19 features the model was trained on, in
the same order, formatted as a list of lists (one inner list per customer).
We compare the model's predictions against

In [5]:
# Grab a few real rows from the test set to send as predictions
import pandas as pd

test_df = pd.read_csv(f"gs://{BUCKET_NAME}/data/test.csv")

# Take 3 sample customers (drop the Churn column since that's what we're predicting)
samples = test_df.drop(columns=["Churn"]).head(3)
actual = test_df["Churn"].head(3).tolist()

# Convert to a list of lists — the format the endpoint expects
instances = samples.values.tolist()

# Send to the endpoint
prediction = endpoint.predict(instances=instances)

print("Predictions:", prediction.predictions)
print("Actual churn labels:", actual)

Predictions: [0.0, 1.0, 0.0]
Actual churn labels: [0, 0, 0]


## Comprehensive Endpoint Demo
This section demonstrates the deployed model the way a real user would
experience it. We take sample customers, show their key characteristics
in human-readable form, send them to the live Vertex AI endpoint, and
display the model's churn prediction alongside the actual outcome.

In [8]:
import pandas as pd

# Load the test set and pull a sample of customers
test_df = pd.read_csv(f"gs://{BUCKET_NAME}/data/test.csv")
sample = test_df.sample(8, random_state=1).reset_index(drop=True)

# Separate features from the actual outcome
features = sample.drop(columns=["Churn"])
actual = sample["Churn"].tolist()

# Send to the live endpoint
prediction = endpoint.predict(instances=features.values.tolist())
predicted = [int(p) for p in prediction.predictions]

# Decode a few key features back to human-readable labels for display
# (these mappings match how LabelEncoder sorted them alphabetically)
contract_map = {0: "Month-to-month", 1: "One year", 2: "Two year"}

# Build a readable results table
results = pd.DataFrame({
    "Tenure (months)": features["tenure"],
    "Monthly Charges": features["MonthlyCharges"],
    "Contract": features["Contract"].map(contract_map),
    "Model Prediction": ["Will churn" if p == 1 else "Will stay" for p in predicted],
    "Actual Outcome": ["Churned" if a == 1 else "Stayed" for a in actual],
    "Correct?": ["✓" if p == a else "✗" for p, a in zip(predicted, actual)]
})

print(results.to_string(index=False))

# Summary accuracy on this sample
correct = sum(1 for p, a in zip(predicted, actual) if p == a)
print(f"\nCorrect on this sample: {correct}/{len(actual)}")

 Tenure (months)  Monthly Charges       Contract Model Prediction Actual Outcome Correct?
               2            20.00 Month-to-month        Will stay         Stayed        ✓
              66            95.30       One year        Will stay         Stayed        ✓
              18            20.10       Two year        Will stay         Stayed        ✓
               8            54.40 Month-to-month        Will stay         Stayed        ✓
              68            25.05       Two year        Will stay         Stayed        ✓
               7            74.35 Month-to-month        Will stay         Stayed        ✓
               2            74.90 Month-to-month       Will churn        Churned        ✓
              16            74.75 Month-to-month       Will churn         Stayed        ✗

Correct on this sample: 7/8
